# Chem 3141 Lab Data Set Modification & Generation Tool

This interactive Jupyter notebook tool allows instructors and teaching assistants to modify master computational data sets (Types A, B, and C) and bulk-generate customized data sets for distribution to a large lab class.

--- 

### Features:
1. **Easy Type Selection**: Switch seamlessly between Dataset Types A, B, and C.
2. **Built-in Master Datasets**: Hard-coded Version 1 master datasets are embedded directly—no external file loading required.
3. **Custom Master Upload**: Easily upload a custom CSV file to serve as the new master baseline.
4. **Interactive Parameter Tuning**: Fine-tune modification values (deltas for pressure, temperature, resistance, rate constants).
5. **Single & Bulk Export**: Preview modified data side-by-side with original master data, and export single CSV files or a ZIP bundle of unique datasets for a student roster.

In [1]:
import io
import os
import zipfile
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, FileLink

# =============================================================================
# 1. EMBEDDED MASTER VERSION 1 DATASETS (Hard-copied)
# =============================================================================

MASTER_A1_TEXT = """Pdial (inHg),left (cm),right (cm),,Patm =,31.20,inHg
26.4,0,0,,T =,21.5,°C
25.2,18,-17,,,,
24.1,32,-31,,,,
23.2,47,-46,,,,
22.1,62,-61,,,,
20.7,78,-77,,,,
19.6,94,-93,,,,
18.5,107,-106,,,,
17.2,122,-122,,,,
16.1,137,-136,,,,
14.9,153,-152,,,,
13.8,168,-167,,,,
12.8,182,-181,,,,
11.6,197,-197,,,,
10.5,211,-211,,,,
9.5,226,-226,,,,
8.1,242,-242,,,,
6.9,257,-256,,,,
5.7,272,-272,,,,
5.0,282,-282,,,,
4.4,289,-289,,,,
3.5,302,-302,,,,
2.2,316,-316,,,,
0.2,331,-331,,,,
"""

MASTER_B1_TEXT = """R (W),T (°C)
5695,5.8
4316,11.4
3316,16.9
2777,20.8
2143,26.6
1736,31.5
1413,36.7
1171,41.2
979,45.9
801.2,51.0
651.8,56.5
555.4,61.0
478.0,65.2
383.1,70.9
330.8,76.0
275.5,81.7
"""

MASTER_C1_TEXT = """T (°C),k (s-1),Dk (s-1)
24.4,0.000073,0.00003
30.0,0.000139,0.00005
34.7,0.000353,0.00007
40.2,0.000717,0.00012
45.1,0.001600,0.00017
50.3,0.003340,0.00015
"""

# =============================================================================
# 2. PARSING, MODIFICATION & EXPORT FUNCTIONS
# =============================================================================

def parse_dataset_A(csv_content):
    """Parse Type A dataset (preserves Patm, T, and (Pdial, left, right) structure)."""
    lines = [line.strip() for line in csv_content.strip().split('\n') if line.strip()]
    parts1 = lines[0].split(',')
    patm_val = float(parts1[5]) if len(parts1) > 5 and parts1[5] else 31.20
    
    parts2 = lines[1].split(',')
    temp_val = float(parts2[5]) if len(parts2) > 5 and parts2[5] else 21.5
    
    rows = []
    for line in lines[1:]:
        p = line.split(',')
        if len(p) >= 3:
            rows.append([float(p[0]), float(p[1]), float(p[2])])
            
    df = pd.DataFrame(rows, columns=['Pdial (inHg)', 'left (cm)', 'right (cm)'])
    meta = {'Patm': patm_val, 'T': temp_val}
    return df, meta

def modify_dataset_A(df, meta, delta_pdial=0.5, delta_left=1.0, delta_right=-1.0, new_temp=None, patm_offset=1.1):
    """Modify Type A dataset parameters."""
    df_mod = df.copy()
    df_mod['Pdial (inHg)'] = (df_mod['Pdial (inHg)'] + delta_pdial).round(1)
    
    new_left = df_mod['left (cm)'].copy()
    new_right = df_mod['right (cm)'].copy()
    
    # Row 1 left and right remain 0
    new_left.iloc[1:] = (new_left.iloc[1:] + delta_left).round(0)
    new_right.iloc[1:] = (new_right.iloc[1:] + delta_right).round(0)
    
    df_mod['left (cm)'] = new_left
    df_mod['right (cm)'] = new_right
    
    temp_out = new_temp if new_temp is not None else meta['T']
    first_pdial = df_mod['Pdial (inHg)'].iloc[0]
    patm_out = round(first_pdial + patm_offset, 2)
    
    meta_mod = {'Patm': patm_out, 'T': round(temp_out, 1)}
    return df_mod, meta_mod

def export_csv_A(df_mod, meta_mod):
    """Format Type A modified data back into original CSV structure."""
    out = []
    out.append(f"Pdial (inHg),left (cm),right (cm),,Patm =,{meta_mod['Patm']:.2f},inHg,")
    r0 = df_mod.iloc[0]
    out.append(f"{r0['Pdial (inHg)']:.1f},{int(r0['left (cm)'])},{int(r0['right (cm)'])},,T =,{meta_mod['T']:.1f},°C,")
    for _, r in df_mod.iloc[1:].iterrows():
        out.append(f"{r['Pdial (inHg)']:.1f},{int(r['left (cm)'])},{int(r['right (cm)'])},,,,,,")
    return "\n".join(out) + "\n"

def parse_dataset_B(csv_content):
    """Parse Type B dataset."""
    df = pd.read_csv(io.StringIO(csv_content.strip()))
    return df, {}

def modify_dataset_B(df, delta_r=-20.0, delta_t=-2.0):
    """Modify Type B dataset parameters."""
    df_mod = df.copy()
    df_mod.iloc[:, 0] = (df_mod.iloc[:, 0] + delta_r).round(1)
    df_mod.iloc[:, 1] = (df_mod.iloc[:, 1] + delta_t).round(1)
    return df_mod, {}

def export_csv_B(df_mod, meta_mod=None):
    """Format Type B modified data into CSV."""
    return df_mod.to_csv(index=False)

def parse_dataset_C(csv_content):
    """Parse Type C dataset."""
    df = pd.read_csv(io.StringIO(csv_content.strip()))
    return df, {}

def modify_dataset_C(df, delta_t=2.0, delta_k=0.000009, delta_dk=0.00005):
    """Modify Type C dataset parameters."""
    df_mod = df.copy()
    df_mod.iloc[:, 0] = (df_mod.iloc[:, 0] + delta_t).round(1)
    df_mod.iloc[:, 1] = (df_mod.iloc[:, 1] + delta_k).round(6)
    df_mod.iloc[:, 2] = (df_mod.iloc[:, 2] + delta_dk).round(5)
    return df_mod, {}

def export_csv_C(df_mod, meta_mod=None):
    """Format Type C modified data into CSV."""
    lines = [f"{df_mod.columns[0]},{df_mod.columns[1]},{df_mod.columns[2]}"]
    for _, r in df_mod.iterrows():
        lines.append(f"{r.iloc[0]:.1f},{r.iloc[1]:.6f},{r.iloc[2]:.5f}")
    return "\n".join(lines) + "\n"

print("Master dataset functions and V1 data structures loaded successfully.")

Master dataset functions and V1 data structures loaded successfully.


In [2]:
# =============================================================================
# 3. INTERACTIVE IPYWIDGETS GUI
# =============================================================================

custom_master_files = {'A': None, 'B': None, 'C': None}

# Type Selection & Master File Management
type_dropdown = widgets.Dropdown(
    options=[('Type A (Manometer/Pressure)', 'A'), ('Type B (Thermistor Resistance)', 'B'), ('Type C (Kinetics Rate Constant)', 'C')],
    value='A',
    description='Data Type:',
    style={'description_width': '140px'}
)

file_uploader = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload Custom Master CSV',
    style={'button_color': '#4a90e2'}
)

reset_master_btn = widgets.Button(
    description='Reset to Default Master V1',
    button_style='warning',
    icon='undo'
)

upload_status_lbl = widgets.HTML(value="<i style='color:gray;'>Using default Master Version 1 data</i>")

# Type A Parameter Controls
a_delta_pdial = widgets.FloatSlider(value=0.5, min=-5.0, max=5.0, step=0.1, description='Δ Pdial (inHg):', style={'description_width': '160px'})
a_delta_left = widgets.FloatSlider(value=1.0, min=-10.0, max=10.0, step=1.0, description='Δ left (cm):', style={'description_width': '160px'})
a_delta_right = widgets.FloatSlider(value=-1.0, min=-10.0, max=10.0, step=1.0, description='Δ right (cm):', style={'description_width': '160px'})
a_temp = widgets.FloatSlider(value=21.5, min=19.5, max=35.0, step=0.1, description='Temperature T (°C):', style={'description_width': '160px'})
a_patm_offset = widgets.FloatSlider(value=1.1, min=-5.0, max=5.0, step=0.1, description='Patm - Pdial[0] (inHg):', style={'description_width': '160px'})

# Type B Parameter Controls
b_delta_r = widgets.FloatSlider(value=-20.0, min=-100.0, max=100.0, step=1.0, description='Δ R (Ω):', style={'description_width': '160px'})
b_delta_t = widgets.FloatSlider(value=-2.0, min=-10.0, max=10.0, step=0.1, description='Δ T (°C):', style={'description_width': '160px'})

# Type C Parameter Controls
c_delta_t = widgets.FloatSlider(value=2.0, min=-10.0, max=10.0, step=0.1, description='Δ T (°C):', style={'description_width': '160px'})
c_delta_k = widgets.BoundedFloatText(value=0.000009, min=-0.001, max=0.001, step=0.000001, format='.6f', description='Δ k (s⁻¹):', style={'description_width': '160px'})
c_delta_dk = widgets.BoundedFloatText(value=0.000050, min=-0.001, max=0.001, step=0.000005, format='.5f', description='Δ Dk (s⁻¹):', style={'description_width': '160px'})

# Preview & Export Controls
out_preview = widgets.Output()
single_export_filename = widgets.Text(value='data_analysis_example_data_sets_A2.csv', description='Export Filename:', style={'description_width': '140px'})
single_export_btn = widgets.Button(description='Export Single CSV', button_style='success', icon='download')
out_single_export = widgets.Output()

# Containers
ctrl_box_A = widgets.VBox([a_delta_pdial, a_delta_left, a_delta_right, a_temp, a_patm_offset])
ctrl_box_B = widgets.VBox([b_delta_r, b_delta_t])
ctrl_box_C = widgets.VBox([c_delta_t, c_delta_k, c_delta_dk])
param_container = widgets.VBox([ctrl_box_A])

def get_master_csv_text(dtype):
    if custom_master_files[dtype] is not None:
        return custom_master_files[dtype]
    if dtype == 'A':
        return MASTER_A1_TEXT
    elif dtype == 'B':
        return MASTER_B1_TEXT
    else:
        return MASTER_C1_TEXT

def update_param_container():
    t = type_dropdown.value
    if t == 'A':
        param_container.children = [ctrl_box_A]
        single_export_filename.value = 'data_analysis_example_data_sets_A2.csv'
    elif t == 'B':
        param_container.children = [ctrl_box_B]
        single_export_filename.value = 'data_analysis_example_data_sets_B2.csv'
    else:
        param_container.children = [ctrl_box_C]
        single_export_filename.value = 'data_analysis_example_data_sets_C2.csv'
        
    if custom_master_files[t] is not None:
        upload_status_lbl.value = f"<b style='color:#2a7ae9;'>Currently using custom uploaded master for Type {t}</b>"
    else:
        upload_status_lbl.value = f"<i style='color:gray;'>Currently using default Master Version 1 data for Type {t}</i>"

def render_preview(*args):
    with out_preview:
        clear_output()
        dtype = type_dropdown.value
        csv_text = get_master_csv_text(dtype)
        
        if dtype == 'A':
            df_orig, meta_orig = parse_dataset_A(csv_text)
            df_mod, meta_mod = modify_dataset_A(
                df_orig, meta_orig, 
                delta_pdial=a_delta_pdial.value, 
                delta_left=a_delta_left.value, 
                delta_right=a_delta_right.value, 
                new_temp=a_temp.value, 
                patm_offset=a_patm_offset.value
            )
            html_str = f"""
            <div style="display:flex; gap:30px;">
                <div style="background-color:#f9f9f9; padding:12px; border-radius:8px; border:1px solid #ddd;">
                    <h4 style="margin-top:0;">Original Master (Type A)</h4>
                    <p><b>Patm:</b> {meta_orig['Patm']:.2f} inHg &nbsp;|&nbsp; <b>T:</b> {meta_orig['T']:.1f} °C</p>
                    {df_orig.to_html(max_rows=10)}
                </div>
                <div style="background-color:#f0f7ff; padding:12px; border-radius:8px; border:1px solid #b6d4fe;">
                    <h4 style="margin-top:0; color:#0d6efd;">Modified Output (Type A)</h4>
                    <p><b>Patm:</b> {meta_mod['Patm']:.2f} inHg &nbsp;|&nbsp; <b>T:</b> {meta_mod['T']:.1f} °C</p>
                    {df_mod.to_html(max_rows=10)}
                </div>
            </div>
            """
            display(HTML(html_str))
            
        elif dtype == 'B':
            df_orig, _ = parse_dataset_B(csv_text)
            df_mod, _ = modify_dataset_B(df_orig, delta_r=b_delta_r.value, delta_t=b_delta_t.value)
            html_str = f"""
            <div style="display:flex; gap:30px;">
                <div style="background-color:#f9f9f9; padding:12px; border-radius:8px; border:1px solid #ddd;">
                    <h4 style="margin-top:0;">Original Master (Type B)</h4>
                    {df_orig.to_html(max_rows=10)}
                </div>
                <div style="background-color:#f0f7ff; padding:12px; border-radius:8px; border:1px solid #b6d4fe;">
                    <h4 style="margin-top:0; color:#0d6efd;">Modified Output (Type B)</h4>
                    {df_mod.to_html(max_rows=10)}
                </div>
            </div>
            """
            display(HTML(html_str))
            
        else:
            df_orig, _ = parse_dataset_C(csv_text)
            df_mod, _ = modify_dataset_C(df_orig, delta_t=c_delta_t.value, delta_k=c_delta_k.value, delta_dk=c_delta_dk.value)
            html_str = f"""
            <div style="display:flex; gap:30px;">
                <div style="background-color:#f9f9f9; padding:12px; border-radius:8px; border:1px solid #ddd;">
                    <h4 style="margin-top:0;">Original Master (Type C)</h4>
                    {df_orig.to_html(max_rows=10)}
                </div>
                <div style="background-color:#f0f7ff; padding:12px; border-radius:8px; border:1px solid #b6d4fe;">
                    <h4 style="margin-top:0; color:#0d6efd;">Modified Output (Type C)</h4>
                    {df_mod.to_html(max_rows=10)}
                </div>
            </div>
            """
            display(HTML(html_str))

def on_type_change(change):
    update_param_container()
    render_preview()

def on_file_upload(change):
    if not file_uploader.value:
        return
    file_info = list(file_uploader.value.values())[0]
    content = file_info['content'].decode('utf-8')
    dtype = type_dropdown.value
    custom_master_files[dtype] = content
    update_param_container()
    render_preview()

def on_reset_master(b):
    dtype = type_dropdown.value
    custom_master_files[dtype] = None
    file_uploader.value.clear()
    update_param_container()
    render_preview()

def on_single_export(b):
    with out_single_export:
        clear_output()
        dtype = type_dropdown.value
        csv_text = get_master_csv_text(dtype)
        filename = single_export_filename.value.strip() or f"dataset_{dtype}_modified.csv"
        
        if dtype == 'A':
            df_orig, meta_orig = parse_dataset_A(csv_text)
            df_mod, meta_mod = modify_dataset_A(df_orig, meta_orig, a_delta_pdial.value, a_delta_left.value, a_delta_right.value, a_temp.value, a_patm_offset.value)
            out_csv = export_csv_A(df_mod, meta_mod)
        elif dtype == 'B':
            df_orig, _ = parse_dataset_B(csv_text)
            df_mod, meta_mod = modify_dataset_B(df_orig, b_delta_r.value, b_delta_t.value)
            out_csv = export_csv_B(df_mod, meta_mod)
        else:
            df_orig, _ = parse_dataset_C(csv_text)
            df_mod, meta_mod = modify_dataset_C(df_orig, c_delta_t.value, c_delta_k.value, c_delta_dk.value)
            out_csv = export_csv_C(df_mod, meta_mod)
            
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(out_csv)
            
        display(HTML(f"<p style='color:green;'><b>Saved successfully:</b> <code>{filename}</code></p>"))
        display(FileLink(filename))

# Event Listeners
type_dropdown.observe(on_type_change, names='value')
file_uploader.observe(on_file_upload, names='value')
reset_master_btn.on_click(on_reset_master)
single_export_btn.on_click(on_single_export)

for widget in [a_delta_pdial, a_delta_left, a_delta_right, a_temp, a_patm_offset, b_delta_r, b_delta_t, c_delta_t, c_delta_k, c_delta_dk]:
    widget.observe(render_preview, names='value')

# Batch Generator Controls
batch_type = widgets.Dropdown(options=[('Type A', 'A'), ('Type B', 'B'), ('Type C', 'C')], value='A', description='Batch Type:', style={'description_width': '140px'})
batch_count = widgets.IntSlider(value=10, min=1, max=100, step=1, description='Student Count (N):', style={'description_width': '140px'})
batch_mode = widgets.RadioButtons(options=[('Randomized Variation per Student', 'random'), ('Stepped Parameter Offset per Student', 'step')], value='random', description='Variation Mode:', style={'description_width': '140px'})
batch_zip_name = widgets.Text(value='chem3141_lab_datasets.zip', description='ZIP Filename:', style={'description_width': '140px'})
batch_export_btn = widgets.Button(description='Generate & Export ZIP', button_style='primary', icon='archive')
out_batch = widgets.Output()

def on_batch_export(b):
    with out_batch:
        clear_output()
        dtype = batch_type.value
        N = batch_count.value
        mode = batch_mode.value
        zip_filename = batch_zip_name.value.strip() or 'class_datasets.zip'
        
        csv_text = get_master_csv_text(dtype)
        zip_buffer = io.BytesIO()
        
        out_dir = "generated_datasets"
        os.makedirs(out_dir, exist_ok=True)
        
        with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
            for i in range(1, N + 1):
                if dtype == 'A':
                    df_orig, meta_orig = parse_dataset_A(csv_text)
                    if mode == 'random':
                        dp = round(float(np.random.uniform(-1.5, 1.5)), 1)
                        dl = float(np.random.choice([-2, -1, 1, 2]))
                        dr = -dl
                        temp = round(float(np.random.uniform(19.5, 35.0)), 1)
                        poff = round(float(np.random.uniform(0.5, 1.8)), 2)
                    else:
                        dp = round(0.1 * i, 1)
                        dl = float(i % 5)
                        dr = -dl
                        temp = round(19.5 + 0.5 * (i % 30), 1)
                        poff = 1.1
                    df_mod, meta_mod = modify_dataset_A(df_orig, meta_orig, dp, dl, dr, temp, poff)
                    out_csv = export_csv_A(df_mod, meta_mod)
                    
                elif dtype == 'B':
                    df_orig, _ = parse_dataset_B(csv_text)
                    if mode == 'random':
                        dr = round(float(np.random.uniform(-40, 40)), 1)
                        dt = round(float(np.random.uniform(-4.0, 4.0)), 1)
                    else:
                        dr = round(-20.0 + 2.0 * i, 1)
                        dt = round(-2.0 + 0.2 * i, 1)
                    df_mod, meta_mod = modify_dataset_B(df_orig, dr, dt)
                    out_csv = export_csv_B(df_mod, meta_mod)
                    
                else:
                    df_orig, _ = parse_dataset_C(csv_text)
                    if mode == 'random':
                        dt = round(float(np.random.uniform(-3.0, 3.0)), 1)
                        dk = round(float(np.random.uniform(-0.00002, 0.00002)), 6)
                        ddk = round(float(np.random.uniform(-0.00008, 0.00008)), 5)
                    else:
                        dt = round(-2.0 + 0.2 * i, 1)
                        dk = round(0.000001 * i, 6)
                        ddk = round(0.000005 * i, 5)
                    df_mod, meta_mod = modify_dataset_C(df_orig, dt, dk, ddk)
                    out_csv = export_csv_C(df_mod, meta_mod)
                
                fname = f"student_{i:02d}_data_type_{dtype}.csv"
                zf.writestr(fname, out_csv)
                
                # Save copy to local directory
                with open(os.path.join(out_dir, fname), 'w', encoding='utf-8') as f:
                    f.write(out_csv)
        
        with open(zip_filename, 'wb') as f:
            f.write(zip_buffer.getvalue())
            
        display(HTML(f"<p style='color:green;'><b>Batch Generation Complete:</b> Generated <b>{N}</b> datasets stored in <code>{out_dir}/</code> and packed into <code>{zip_filename}</code></p>"))
        display(FileLink(zip_filename))

batch_export_btn.on_click(on_batch_export)

# Tab Construction
tab1_layout = widgets.VBox([
    widgets.HTML("<h3>1. Master Data Source</h3>"),
    type_dropdown,
    widgets.HBox([file_uploader, reset_master_btn]),
    upload_status_lbl,
    widgets.HTML("<hr><h3>2. Numerical Parameter Controls</h3>"),
    param_container,
    widgets.HTML("<hr><h3>3. Data Comparison Preview & Single Export</h3>"),
    out_preview,
    widgets.HBox([single_export_filename, single_export_btn]),
    out_single_export
])

tab2_layout = widgets.VBox([
    widgets.HTML("<h3>Batch Class Roster Exporter</h3><p>Bulk-generate N distinct datasets for your class roster with unique variations.</p>"),
    batch_type,
    batch_count,
    batch_mode,
    batch_zip_name,
    batch_export_btn,
    out_batch
])

main_tabs = widgets.Tab(children=[tab1_layout, tab2_layout])
main_tabs.set_title(0, 'Single Dataset Modifier')
main_tabs.set_title(1, 'Batch Class Roster Exporter')

# Display App
update_param_container()
render_preview()
display(main_tabs)